In [6]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
import requests
import time
from io import StringIO

# I. Load DPCFamB Mcs Sequences

In [7]:
# DPCFamB Mcs Sequences
df1 = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/DPCFamB/dpcfamB_sequences.csv")
df1.head()

,mcid,protein_id,seq_range,seq_length,aa_seq
0,MC113651,A0A0K6GCJ4,2-376,375,PTVVSSRALVPIGRRIRNAPEDSDASKAIVLRRKQQGETEDLSKAL...
1,MC113651,A8N621,40-384,345,TRALILRNGKHGAMGTGEIIASNKITGREKLDLLAEELIEKMKTAV...
2,MC113651,A0A0H2RRJ7,98-399,302,SNAVRAPFDLDSLIRWSQSQMDAALDQVNNLHETDFCYEIVERHIS...
3,MC113651,G4TJX7,26-369,344,VILSNQRREFVNTSSNAIALRYEPHGIWGEGQLASFKKRSGQEKLA...
4,MC113651,UPI0004449C2E,4-413,410,PPKDSKQPKDAPASTSRDLVRVGPRHVSAKHVVGRRNEDGQKPTTA...


In [8]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200826 entries, 0 to 1200825
Data columns (total 5 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   mcid        1200826 non-null  object
 1   protein_id  1200826 non-null  object
 2   seq_range   1200826 non-null  object
 3   seq_length  1200826 non-null  int64 
 4   aa_seq      1200826 non-null  object
dtypes: int64(1), object(4)
memory usage: 45.8+ MB


# I. Check which protein IDs already exist in DPCFam dataframe source

In [9]:
# We use the DPCfam dataframe with info about proteins in UniProtKB 
df2 = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/uniref50_proteins.csv")
df2.head()

,uniref50_id,uniprotkb_id,uniprotkb_accession,length
0,Q8WZ42-5,Q8WZ42-5,Q8WZ42-5,32900
1,Q3ASY8,Q3ASY8_CHLCH,Q3ASY8,36805
2,G5B0U1,G5B0U1_HETGA,G5B0U1,36507
3,Q8WZ42,TITIN_HUMAN,Q9Y6L9,34350
4,K7EE71,K7EE71_ORNAN,K7EE71,7472


In [10]:
# Info
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23531980 entries, 0 to 23531979
Data columns (total 4 columns):
 #   Column               Dtype 
---  ------               ----- 
 0   uniref50_id          object
 1   uniprotkb_id         object
 2   uniprotkb_accession  object
 3   length               int64 
dtypes: int64(1), object(3)
memory usage: 718.1+ MB


In [11]:
# How many unique protein IDs are there in uniref50_id column?
print("Number of unique protein IDs in uniref50_id column:", df2["uniref50_id"].nunique())
# How many unique protein IDs are there in uniprotkb_accession column?
print("Number of unique protein IDs in uniprotkb_accession column:", df2["uniprotkb_accession"].nunique())
# How many unique protein IDs are there in their intersection?
intersection = set(df2["uniref50_id"]).intersection(set(df2["uniprotkb_accession"]))
print("Number of unique protein IDs in the intersection of uniref50_id and uniprotkb_accession columns:", len(intersection))
# Print the first 5 unique protein IDs in the intersection
print("First 5 unique protein IDs in the intersection of uniref50_id and uniprotkb_accession columns:", list(intersection)[:5])
# How many samples are there in their difference?
difference = set(df2["uniref50_id"]).difference(set(df2["uniprotkb_accession"]))
print("Number of unique protein IDs in the difference between uniref50_id and uniprotkb_accession columns:", len(difference))
# Print the first 5 unique protein IDs in the difference
print("First 5 unique protein IDs in the difference between uniref50_id and uniprotkb_accession columns:", list(difference)[:5])

Number of unique protein IDs in uniref50_id column: 23531980
Number of unique protein IDs in uniprotkb_accession column: 19681201
Number of unique protein IDs in the intersection of uniref50_id and uniprotkb_accession columns: 19588018
First 5 unique protein IDs in the intersection of uniref50_id and uniprotkb_accession columns: ['M9XAY3', 'A0A0Q9YPD5', 'A0A0V0HQV7', 'A0A1G6YID9', 'V7C4A2']
Number of unique protein IDs in the difference between uniref50_id and uniprotkb_accession columns: 3943962
First 5 unique protein IDs in the difference between uniref50_id and uniprotkb_accession columns: ['UPI00055691B0', 'UPI00044C3476', 'UPI000873E538', 'UPI000625A089', 'A0A0M4RHM0']


**1. We hypothesise that DPCFamB proteins IDs must exist in our existing dataframe df2**

In [12]:
# Now, let's consider protein IDs in df1 and check how many of them are in the intersection of protein IDs in df2
protein_ids_df1 = set(df1["protein_id"])
print("Number of unique protein IDs in df1:", len(protein_ids_df1))
protein_ids_uniref50 = set(df2["uniref50_id"])
protein_ids_uniprotkb = set(df2["uniprotkb_accession"])
intersection = protein_ids_uniref50.intersection(protein_ids_uniprotkb)
# df1 inter Uniref50
intersection_df1 = protein_ids_df1.intersection(intersection)
print("Number of unique protein IDs in df1 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2:", len(intersection_df1))
# Print the first 5 unique protein IDs in the intersection of df1 and the intersection of uniref50_id and uniprotkb_accession columns in df2    
print("First 5 unique protein IDs in df1 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2:", list(intersection_df1)[:5])

Number of unique protein IDs in df1: 1014273
Number of unique protein IDs in df1 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2: 821785
First 5 unique protein IDs in df1 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2: ['S0KK60', 'A0A091N1H3', 'Q67QH0', 'A0A0B0EQI4', 'B9L8D0']


**2. We compare protein IDs in DPCFamB with Uniref50**

In [13]:
# How many protein IDs in df1 are in uniref50_id column in df2?
intersection_df1_uniref50 = protein_ids_df1.intersection(set(df2["uniref50_id"]))
print("Number of unique protein IDs in df1 that are in uniref50_id column in df2:", len(intersection_df1_uniref50))
# Print the first 5 unique protein IDs in the intersection of df1 and uniref50_id column in df2    
print("First 5 unique protein IDs in df1 that are in uniref50_id column in df2:", list(intersection_df1_uniref50)[:5])

Number of unique protein IDs in df1 that are in uniref50_id column in df2: 1014273
First 5 unique protein IDs in df1 that are in uniref50_id column in df2: ['S0KK60', 'A0A091N1H3', 'Q67QH0', 'A0A0B0EQI4', 'B9L8D0']


**Conclusion**:

`All DPCFamB protein IDs already exist in our source base, meaninng we know their lengths`.

In [14]:
# What about DPCFam Standard? 
df3 = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/dpcfam_sequences.csv")
df3.head()

,mcid,protein_id,seq_range,seq_length,aa_seq
0,MC82027,E8W9U6,2-131,130,ILLLSTSDTDLLSARAAQGPVSYRYANPSRLSLDALPQLLEGTDLV...
1,MC82027,I7DYJ9,1-143,143,MHVVFRESHGLDESDTPQDLGQTPADLVVLSFSDSDLGAFAAGWHR...
2,MC82027,A0A0M6ZLI4,13-175,163,DGGDAVDLGQAPADILFLSAADTELGSFSAAHSALGTDSASLRLAN...
3,MC82027,A0A1X4ITI5,1-182,182,MHLLAATPGNLDDGLEPVDLGQSPADLVVLSAADTDLAALSEARNL...
4,MC82027,A0A1Q7WDG5,5-140,136,FVLLSTADTDLLAARASGAPWRVANPTRLGAADLPALLDGATLAVV...


In [15]:
# Now, let's consider protein IDs in df3 and check how many of them are in the intersection of protein IDs in df2
protein_ids_df3 = set(df3["protein_id"])
print("Number of unique protein IDs in df3:", len(protein_ids_df3))
protein_ids_uniref50 = set(df2["uniref50_id"])
protein_ids_uniprotkb = set(df2["uniprotkb_accession"])
intersection = protein_ids_uniref50.intersection(protein_ids_uniprotkb)
# df3 inter Uniref50
intersection_df3 = protein_ids_df3.intersection(intersection)
print("Number of unique protein IDs in df3 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2:", len(intersection_df3))
# Print the first 5 unique protein IDs in the intersection of df3 and the intersection of uniref50_id and uniprotkb_accession columns in df2    
print("First 5 unique protein IDs in df3 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2:", list(intersection_df3)[:5])

Number of unique protein IDs in df3: 10027691
Number of unique protein IDs in df3 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2: 8271181
First 5 unique protein IDs in df3 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2: ['A0A0G1Q6D1', 'M9XAY3', 'A0A1R0IUZ4', 'A0A0Q9YPD5', 'K9QVM5']


In [16]:
# How many protein IDs in df3 are in uniref50_id column in df2?
intersection_df3_uniref50 = protein_ids_df3.intersection(protein_ids_uniref50)
print("Number of unique protein IDs in df3 that are in uniref50_id column in df2:", len(intersection_df3_uniref50))
# Print the first 5 unique protein IDs in the intersection of df3 and uniref50_id column in df2    
print("First 5 unique protein IDs in df3 that are in uniref50_id column in df2:", list(intersection_df3_uniref50)[:5])

Number of unique protein IDs in df3 that are in uniref50_id column in df2: 10027691
First 5 unique protein IDs in df3 that are in uniref50_id column in df2: ['A0A0G1Q6D1', 'A0A0M4RHM0', 'M9XAY3', 'A0A1R0IUZ4', 'A0A0Q9YPD5']


Great, all DPCFam protein ids are known in UniRef50. We have their lengths.

In [17]:
# Based on MCIDs, how many unique protein IDs are there in df1 and df3?
unique_mcs_ids_df1 = set(df1["mcid"])
unique_mcs_ids_df3 = set(df3["mcid"])
intersection_df1_df3 = unique_mcs_ids_df1.intersection(unique_mcs_ids_df3)
print("Number of unique protein IDs in df1:", len(unique_mcs_ids_df1))
print("Number of unique protein IDs in df3:", len(unique_mcs_ids_df3))
print("Number of unique protein IDs in the intersection of df1 and df3:", len(intersection_df1_df3))
# Print the first 5 unique protein IDs in the intersection of df1 and df3
print("First 5 unique protein IDs in the intersection of df1 and df3:", list(intersection_df1_df3)[:5])

Number of unique protein IDs in df1: 34556
Number of unique protein IDs in df3: 46828
Number of unique protein IDs in the intersection of df1 and df3: 0
First 5 unique protein IDs in the intersection of df1 and df3: []


Great, DPCFam Standard and DPCFamB are disjoints. We can merge them with easy.